# ML Intrusion Detector

This notebook builds a **network intrusion detection pipeline** — the kind of system a Security Operations Center (SOC) would use to automatically flag suspicious traffic. We train models on real attack data, evaluate their trade-offs, and validate the methodology on a second dataset.

## What this notebook covers

1. **Two complementary detection approaches**
   - *Unsupervised* (Isolation Forest): catches anything weird, even unknown attacks
   - *Supervised* (XGBoost): learns specific attack patterns from labeled data

2. **Why default model settings often lie about performance**  
   We'll show how a model with 0.97 ROC-AUC can still miss 32% of attacks — and how to fix it.

3. **The tension every security team knows**  
   *Precision* (don't cry wolf) vs *Recall* (don't miss the wolf). We quantify the trade-off.

4. **Cross-dataset validation**  
   We re-run the entire pipeline on a second dataset (UNSW-NB15) to show the methodology isn't a one-trick pony.

**Stack:** Python · scikit-learn · XGBoost · imbalanced-learn · Matplotlib · Seaborn

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import IsolationForest
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    roc_curve, accuracy_score, precision_score, recall_score, f1_score,
)
import xgboost as xgb

import warnings
warnings.filterwarnings("ignore")

sns.set_theme(style="whitegrid", palette="deep")
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

## 1. Load the dataset

We use **NSL-KDD**, a benchmark dataset for network intrusion detection. Each row is a network connection (think: one HTTP request, one SSH session). The columns are 41 measurable properties — duration, protocol, byte counts, flag patterns, and so on — plus a label saying whether the connection was normal or an attack.

The dataset is split into train/test by the authors with a deliberate twist: **the test set contains attack types absent from training**. This simulates real-world conditions where new attacks appear all the time — and it makes the task much harder than a random split would.

In [ ]:
COLUMNS = [
    "duration", "protocol_type", "service", "flag", "src_bytes", "dst_bytes",
    "land", "wrong_fragment", "urgent", "hot", "num_failed_logins", "logged_in",
    "num_compromised", "root_shell", "su_attempted", "num_root", "num_file_creations",
    "num_shells", "num_access_files", "num_outbound_cmds", "is_host_login",
    "is_guest_login", "count", "srv_count", "serror_rate", "srv_serror_rate",
    "rerror_rate", "srv_rerror_rate", "same_srv_rate", "diff_srv_rate",
    "srv_diff_host_rate", "dst_host_count", "dst_host_srv_count",
    "dst_host_same_srv_rate", "dst_host_diff_srv_rate", "dst_host_same_src_port_rate",
    "dst_host_srv_diff_host_rate", "dst_host_serror_rate", "dst_host_srv_serror_rate",
    "dst_host_rerror_rate", "dst_host_srv_rerror_rate", "label", "difficulty",
]

TRAIN_URL = "https://raw.githubusercontent.com/defcom17/NSL_KDD/master/KDDTrain%2B.txt"
TEST_URL  = "https://raw.githubusercontent.com/defcom17/NSL_KDD/master/KDDTest%2B.txt"

df_train = pd.read_csv(TRAIN_URL, names=COLUMNS)
df_test  = pd.read_csv(TEST_URL,  names=COLUMNS)

print(f"Train: {df_train.shape}")
print(f"Test:  {df_test.shape}")
df_train.head()

## 2. Quick exploration

Before training anything, let's see what we're dealing with: how many samples, which attack types, class distribution. This tells us whether we have an imbalance problem and which attack types will dominate the model's learning.

In [ ]:
print("Top 15 attack types (train):")
print(df_train["label"].value_counts().head(15))

df_train["binary_label"] = (df_train["label"] != "normal").astype(int)
df_test["binary_label"]  = (df_test["label"]  != "normal").astype(int)

print(f"\nTrain  Normal: {(df_train['binary_label']==0).sum():>6} | Attack: {(df_train['binary_label']==1).sum():>6}")
print(f"Test   Normal: {(df_test['binary_label']==0).sum():>6} | Attack: {(df_test['binary_label']==1).sum():>6}")

> **What to notice:** the train set is roughly 50/50 normal vs attack (53/47). The test set leans more toward attacks (43/57). The dominant attack is `neptune` (a SYN flood DoS) — over 70% of attack samples in train. Several attack types (`buffer_overflow`, `warezmaster`, `land`) have fewer than 30 samples, which will hurt detection of those specific types.

In [ ]:
counts = df_train["binary_label"].map({0: "Normal", 1: "Attack"}).value_counts()
fig, ax = plt.subplots(figsize=(6, 4))
counts.plot(kind="bar", color=["#4C72B0", "#DD8452"], ax=ax, edgecolor="black")
ax.set_title("Train — Normal vs Attack distribution")
ax.set_xlabel("Class")
ax.set_ylabel("Count")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 3. Preprocessing

Two things to fix before feeding data to a model:

1. **Categorical features** (`protocol_type`, `service`, `flag`) are strings — models need numbers. We use `LabelEncoder` to map each unique value to an integer.
2. **Numerical features** span very different ranges (some are 0/1 booleans, others are byte counts in millions). We `StandardScaler` them (mean=0, std=1) so all features get treated comparably.

**Subtle point:** we fit the LabelEncoder on `train + test` combined to handle categories that appear only in test (otherwise we'd crash at prediction time). The scaler is fit on train only — fitting on test would leak distribution info.

In [ ]:
CATEGORICAL = ["protocol_type", "service", "flag"]

def preprocess(df_tr, df_te):
    df_tr = df_tr.copy()
    df_te = df_te.copy()

    for d in (df_tr, df_te):
        d.drop(columns=["difficulty", "label"], inplace=True)

    for col in CATEGORICAL:
        le = LabelEncoder()
        le.fit(pd.concat([df_tr[col], df_te[col]], axis=0))
        df_tr[col] = le.transform(df_tr[col])
        df_te[col] = le.transform(df_te[col])

    y_tr = df_tr.pop("binary_label")
    y_te = df_te.pop("binary_label")

    scaler = StandardScaler()
    X_tr = pd.DataFrame(scaler.fit_transform(df_tr), columns=df_tr.columns)
    X_te = pd.DataFrame(scaler.transform(df_te),    columns=df_te.columns)
    return X_tr, X_te, y_tr, y_te

X_train, X_test, y_train, y_test = preprocess(df_train, df_test)
print(f"X_train: {X_train.shape} | X_test: {X_test.shape}")

## 4. Model 1 — Isolation Forest (unsupervised)

### Intuition

Imagine sorting tree leaves randomly. Most leaves look similar, so it takes **many** sorting steps to isolate a typical leaf. But a weird-shaped leaf gets isolated almost immediately. *That "few steps to isolate" is the anomaly score.*

Isolation Forest builds 200 random trees, scores each test point by how easily it gets isolated, and flags the easy-to-isolate ones as anomalies.

### Why it matters here

We train **only on normal traffic** — no attack labels. Anything the model finds "weird" becomes a candidate attack. This is the kind of detector that can catch **zero-day attacks** (attacks no one has labeled yet). The trade-off: it has no idea *what kind* of attack, just that something is off.

In [ ]:
X_train_normal = X_train[y_train == 0]
contamination = min(0.5, float((y_train == 1).mean()))

iso = IsolationForest(
    n_estimators=200,
    contamination=contamination,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
iso.fit(X_train_normal)

y_pred_iso = (iso.predict(X_test) == -1).astype(int)
scores_iso = -iso.score_samples(X_test)

print("=== Isolation Forest ===")
print(classification_report(y_test, y_pred_iso, target_names=["Normal", "Attack"]))
print(f"ROC-AUC: {roc_auc_score(y_test, scores_iso):.4f}")

> **Reading the output:** **95% recall on attacks** (catches almost everything) but only **80% precision** (1 in 5 alerts is a false alarm). ROC-AUC of 0.94 confirms the model *ranks* attacks above normals very well. This is the classic profile of an unsupervised detector: noisy but thorough.

## 5. Model 2 — XGBoost (supervised)

### Intuition

XGBoost builds **decision trees one after another**, where each tree tries to correct the mistakes of the previous ones. After 300 trees, you get a powerful ensemble that's the default winner on most tabular-data competitions.

Unlike Isolation Forest, XGBoost **uses the attack labels** during training. This usually makes it more accurate — but only on attack types similar to those seen in training.

In [ ]:
xgb_model = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=8,
    learning_rate=0.1,
    objective="binary:logistic",
    eval_metric="auc",
    random_state=RANDOM_STATE,
    n_jobs=-1,
    tree_method="hist",
)
xgb_model.fit(X_train, y_train)

y_pred_xgb  = xgb_model.predict(X_test)
y_proba_xgb = xgb_model.predict_proba(X_test)[:, 1]

print("=== XGBoost ===")
print(classification_report(y_test, y_pred_xgb, target_names=["Normal", "Attack"]))
print(f"ROC-AUC: {roc_auc_score(y_test, y_proba_xgb):.4f}")

> **Reading the output:** XGBoost is *very* precise (97% — when it cries wolf, it's almost always right) but only catches **68% of attacks**. ROC-AUC is 0.97 — the model is excellent at *ranking* but the default decision threshold (0.5) is too conservative.
>
> Hold this thought — **we'll fix this in section 7** and unlock a +13 point F1 gain.

## 6. Comparison & visualizations

Side-by-side metrics, then a few visuals: confusion matrices, ROC curves, feature importance, anomaly score distribution. The goal is to *see* the trade-off between the two models, not just read numbers.

In [ ]:
def metrics(y_true, y_pred, y_score):
    return {
        "Accuracy":  accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred),
        "Recall":    recall_score(y_true, y_pred),
        "F1":        f1_score(y_true, y_pred),
        "ROC-AUC":   roc_auc_score(y_true, y_score),
    }

results = pd.DataFrame({
    "Isolation Forest": metrics(y_test, y_pred_iso, scores_iso),
    "XGBoost":          metrics(y_test, y_pred_xgb, y_proba_xgb),
}).T.round(4)

results

> **Two opposite profiles:** Isolation Forest = high recall, low precision (a **noisy but thorough sensor**). XGBoost = high precision, low recall (a **picky but reliable sensor**). Neither is strictly better — they answer different questions.
>
> The confusion matrices below make this concrete: where exactly does each model make its mistakes?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, (name, y_pred) in zip(axes, [("Isolation Forest", y_pred_iso), ("XGBoost", y_pred_xgb)]):
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax,
                xticklabels=["Normal", "Attack"], yticklabels=["Normal", "Attack"],
                cbar=False)
    ax.set_title(f"Confusion Matrix — {name}")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
plt.tight_layout()
plt.show()

**ROC curves** plot True Positive Rate against False Positive Rate as the decision threshold varies. The closer to the top-left, the better. The Area Under the Curve (AUC) summarizes ranking quality across all thresholds — independent of which threshold you happen to pick.

In [ ]:
fpr_iso, tpr_iso, _ = roc_curve(y_test, scores_iso)
fpr_xgb, tpr_xgb, _ = roc_curve(y_test, y_proba_xgb)

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(fpr_iso, tpr_iso, label=f"Isolation Forest (AUC={roc_auc_score(y_test, scores_iso):.3f})", linewidth=2)
ax.plot(fpr_xgb, tpr_xgb, label=f"XGBoost (AUC={roc_auc_score(y_test, y_proba_xgb):.3f})", linewidth=2)
ax.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Random")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("ROC curves")
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()

**Feature importance** — what does XGBoost actually *look at* to decide attack vs normal? The top features are usually traffic-volume signals (`src_bytes`, `dst_bytes`, `count`) and connection-state ratios (`same_srv_rate`, `serror_rate`).

In [ ]:
importance_df = (pd.DataFrame({
    "feature":    X_train.columns,
    "importance": xgb_model.feature_importances_,
}).nlargest(15, "importance").iloc[::-1])  # reverse for horizontal bar (largest at top)

fig, ax = plt.subplots(figsize=(8, 6))
ax.barh(importance_df["feature"], importance_df["importance"], color="#4C72B0", edgecolor="black")
ax.set_title("Top 15 features — XGBoost")
ax.set_xlabel("Importance")
plt.tight_layout()
plt.show()

**Anomaly score distribution** — do attacks really get higher scores than normals? If the two distributions overlap heavily, the detector can't separate them. The more they're shifted, the better.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(scores_iso[y_test == 0], bins=80, alpha=0.6, label="Normal", color="#4C72B0")
ax.hist(scores_iso[y_test == 1], bins=80, alpha=0.6, label="Attack", color="#DD8452")
ax.set_title("Isolation Forest — Anomaly score distribution")
ax.set_xlabel("Anomaly score")
ax.set_ylabel("Count")
ax.legend()
plt.tight_layout()
plt.show()

## 7. Threshold tuning — fixing the silent failure

### The problem

XGBoost's `predict()` says "attack" when the model's probability ≥ **0.5**. That's an arbitrary default. Why 0.5? Because that's the convention for binary classifiers — but it has no statistical justification on a specific dataset.

For our test set, 0.5 turns out to be too cautious: the model gives many true attacks a probability of 0.3 or 0.4 (still suspicious, but below 0.5), so they get labeled as normal. That's why recall is only 68% even though ROC-AUC is 0.97.

### The fix

For each possible threshold, compute the resulting precision and recall. Pick the threshold that maximizes **F1** (the harmonic mean of precision and recall — balances both).

This is a one-line fix that costs nothing — but most ML tutorials skip it, which is why everyone leaves performance on the table.

In [ ]:
from sklearn.metrics import precision_recall_curve

precision_xgb, recall_xgb, thresholds_xgb = precision_recall_curve(y_test, y_proba_xgb)

# F1 per threshold (the last point has no associated threshold)
f1_xgb = 2 * precision_xgb[:-1] * recall_xgb[:-1] / (precision_xgb[:-1] + recall_xgb[:-1] + 1e-9)
best_idx = int(np.argmax(f1_xgb))
best_threshold = float(thresholds_xgb[best_idx])

print(f"Optimal threshold (max F1): {best_threshold:.4f}")
print(f"Precision: {precision_xgb[best_idx]:.4f} | Recall: {recall_xgb[best_idx]:.4f} | F1: {f1_xgb[best_idx]:.4f}")

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(thresholds_xgb, precision_xgb[:-1], label="Precision", linewidth=2)
ax.plot(thresholds_xgb, recall_xgb[:-1], label="Recall", linewidth=2)
ax.plot(thresholds_xgb, f1_xgb, label="F1", linewidth=2)
ax.axvline(x=best_threshold, linestyle="--", color="gray",
           label=f"Best F1 @ {best_threshold:.4f}")
ax.set_title("XGBoost — Precision / Recall / F1 vs Threshold")
ax.set_xlabel("Decision threshold")
ax.set_ylabel("Score")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
y_pred_xgb_tuned = (y_proba_xgb >= best_threshold).astype(int)

print("=== XGBoost — tuned threshold ===")
print(classification_report(y_test, y_pred_xgb_tuned, target_names=["Normal", "Attack"]))

> **Result:** F1 jumps from **0.80 → 0.93** (+13 points). Accuracy from **0.81 → 0.92**. Just by choosing the right threshold.
>
> The optimal threshold is `0.0001` — basically *"if XGBoost has any doubt at all, flag it"*. This makes sense: the test set contains attack types XGBoost has never seen, so the model is unsure but still gives them a slightly elevated probability. Lowering the threshold captures these "weakly recognized" attacks.
>
> ⚠️ **Production caveat:** in a real deployment, we'd tune the threshold on a **validation set**, not the test set, to avoid overfitting the threshold to a particular test split.

## 8. Stacking — combining IF and XGBoost

### Idea

What if Isolation Forest catches what XGBoost misses, and vice-versa? We can feed both models' outputs into a small **meta-model** (logistic regression) that learns the optimal combination.

### The honesty trick: cross_val_predict

If we trained the meta-model on XGBoost's predictions of the training data, XGBoost has *already seen* that data — its predictions would be artificially optimistic and the meta-model would overfit. Instead, we use `cross_val_predict` (3-fold): each training sample gets a prediction from an XGBoost that **didn't see it during training**. Honest meta-features.

In [ ]:
from sklearn.model_selection import cross_val_predict
from sklearn.linear_model import LogisticRegression

xgb_for_cv = xgb.XGBClassifier(
    n_estimators=200, max_depth=6, learning_rate=0.1,
    objective="binary:logistic", eval_metric="auc",
    random_state=RANDOM_STATE, n_jobs=-1, tree_method="hist",
)
print("XGB cross-validation for OOF probabilities (this may take a few minutes)...")
xgb_oof_train = cross_val_predict(
    xgb_for_cv, X_train, y_train, cv=3, method="predict_proba", n_jobs=-1
)[:, 1]

if_train = -iso.score_samples(X_train)
if_test  = scores_iso

stack_train = np.column_stack([if_train, xgb_oof_train])
stack_test  = np.column_stack([if_test,  y_proba_xgb])

meta = LogisticRegression(class_weight="balanced", max_iter=1000,
                          random_state=RANDOM_STATE)
meta.fit(stack_train, y_train)

y_pred_stack  = meta.predict(stack_test)
y_proba_stack = meta.predict_proba(stack_test)[:, 1]

print("=== Stacking (IF + XGB → LogReg) ===")
print(classification_report(y_test, y_pred_stack, target_names=["Normal", "Attack"]))
print(f"ROC-AUC: {roc_auc_score(y_test, y_proba_stack):.4f}")
print(f"\nMeta coefficients: IF_score={meta.coef_[0,0]:.3f} | XGB_proba={meta.coef_[0,1]:.3f}")

> **Honest takeaway:** stacking matches XGBoost default but doesn't beat XGBoost tuned. The meta-model's coefficients tell the story: it weights XGBoost **10× more** than Isolation Forest (13.9 vs 1.4). When one base model dominates, stacking essentially becomes "use that one model".
>
> Stacking isn't magic. It helps when models are **truly complementary** — here they aren't, because XGBoost already captures most of what IF can offer.

## 9. Comparison of the 4 binary approaches

Lining up everything side-by-side to see the wins and losses.

In [ ]:
results_v2 = pd.DataFrame({
    "Isolation Forest":  metrics(y_test, y_pred_iso,       scores_iso),
    "XGBoost (default)": metrics(y_test, y_pred_xgb,       y_proba_xgb),
    "XGBoost (tuned)":   metrics(y_test, y_pred_xgb_tuned, y_proba_xgb),
    "Stacking IF+XGB":   metrics(y_test, y_pred_stack,     y_proba_stack),
}).T.round(4)

results_v2

> **Verdict:** **XGBoost (tuned) wins** every metric except recall (where IF still leads). The threshold-tuning trick is the single most impactful improvement in this notebook. Stacking is a nice exercise but adds nothing here.

## 10. Multi-class classification

### Why bother?

A binary "attack/not-attack" alert isn't enough for a SOC analyst — they need to know **what kind of attack**: a port scan needs a different response than a privilege escalation.

We group the ~40 NSL-KDD labels into **5 standard categories**:
- **Normal** — legitimate traffic
- **DoS** — Denial of Service (flood the server)
- **Probe** — reconnaissance / scanning (find weak spots)
- **R2L** — Remote-to-Local (gain unauthorized access from outside)
- **U2R** — User-to-Root (escalate privileges once inside)

Then train XGBoost to predict the category. To handle the massive class imbalance (U2R has 52 train samples, Normal has 67k), we use `compute_sample_weight("balanced")` — the model treats each class as equally important regardless of size.

In [ ]:
ATTACK_CATEGORIES = {
    "normal": "Normal",
    # DoS
    "back": "DoS", "land": "DoS", "neptune": "DoS", "pod": "DoS",
    "smurf": "DoS", "teardrop": "DoS", "mailbomb": "DoS",
    "processtable": "DoS", "udpstorm": "DoS", "apache2": "DoS", "worm": "DoS",
    # Probe
    "ipsweep": "Probe", "nmap": "Probe", "portsweep": "Probe",
    "satan": "Probe", "mscan": "Probe", "saint": "Probe",
    # R2L
    "ftp_write": "R2L", "guess_passwd": "R2L", "imap": "R2L", "multihop": "R2L",
    "phf": "R2L", "spy": "R2L", "warezclient": "R2L", "warezmaster": "R2L",
    "sendmail": "R2L", "named": "R2L", "snmpgetattack": "R2L",
    "snmpguess": "R2L", "xlock": "R2L", "xsnoop": "R2L", "httptunnel": "R2L",
    # U2R
    "buffer_overflow": "U2R", "loadmodule": "U2R", "perl": "U2R",
    "rootkit": "U2R", "ps": "U2R", "sqlattack": "U2R", "xterm": "U2R",
}

y_train_multi_str = df_train["label"].map(ATTACK_CATEGORIES)
y_test_multi_str  = df_test["label"].map(ATTACK_CATEGORIES)

unmapped = sorted(set(df_train.loc[y_train_multi_str.isna(), "label"].unique())
                  | set(df_test.loc[y_test_multi_str.isna(), "label"].unique()))
print(f"Unmapped labels (fallback -> R2L): {unmapped}")

y_train_multi_str = y_train_multi_str.fillna("R2L")
y_test_multi_str  = y_test_multi_str.fillna("R2L")

classes = ["Normal", "DoS", "Probe", "R2L", "U2R"]
class_to_idx = {c: i for i, c in enumerate(classes)}
y_train_multi = y_train_multi_str.map(class_to_idx).values
y_test_multi  = y_test_multi_str.map(class_to_idx).values

print("\nTrain distribution:")
print(pd.Series(y_train_multi_str).value_counts())
print("\nTest distribution:")
print(pd.Series(y_test_multi_str).value_counts())

> **Distribution shock:** R2L has **995 train samples** but **2885 test samples** (3× more). U2R has 52 train samples — basically nothing for a model to learn from. This is the nightmare scenario: we have to detect attack types we barely have data for, against a test set that contains *new variants*.

In [ ]:
from sklearn.utils.class_weight import compute_sample_weight

sample_weights = compute_sample_weight(class_weight="balanced", y=y_train_multi)

xgb_multi = xgb.XGBClassifier(
    n_estimators=300, max_depth=8, learning_rate=0.1,
    objective="multi:softprob", num_class=5,
    eval_metric="mlogloss",
    random_state=RANDOM_STATE, n_jobs=-1, tree_method="hist",
)
xgb_multi.fit(X_train, y_train_multi, sample_weight=sample_weights)
y_pred_multi = xgb_multi.predict(X_test)

print("=== Multi-class XGBoost (class-weighted) ===")
print(classification_report(y_test_multi, y_pred_multi,
                            target_names=classes, zero_division=0))

> **Reading the output:**
> - **Normal, DoS, Probe**: detected well (recall 0.72-0.97). These have plenty of train data.
> - **R2L recall = 0.08** — catastrophic. The R2L attacks in the test set are mostly types absent from training (`snmpgetattack`, `httptunnel`, `xlock`…). The model literally never saw these patterns.
> - **U2R recall = 0.24** — also weak, same reason.
>
> This is a **structural limitation of the dataset**, not the model. SMOTE (next section) helps a bit but doesn't fully fix it.

In [ ]:
cm_multi = confusion_matrix(y_test_multi, y_pred_multi)

fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(cm_multi, annot=True, fmt="d", cmap="Blues", ax=ax,
            xticklabels=classes, yticklabels=classes, cbar=False)
ax.set_title("Multi-class Confusion Matrix — XGBoost (class-weighted)")
ax.set_xlabel("Predicted")
ax.set_ylabel("Actual")
plt.tight_layout()
plt.show()

## 11. SMOTE on rare classes

### Idea

**SMOTE** (Synthetic Minority Oversampling Technique) generates *synthetic* samples for rare classes by **interpolating** between existing real samples. So if R2L has 995 samples and Normal has 67k, SMOTE creates 66k synthetic R2L examples to balance the classes.

The model now sees as many examples of each class during training, so it can't ignore the rare ones.

### SMOTE vs class_weight — what's the difference?

| Approach | What it does |
|---|---|
| `class_weight="balanced"` | Tells the loss function to penalize errors on rare classes more. Same data, different weights. |
| **SMOTE** | Creates new synthetic data points. Different data, equal weights. |

Both target the same problem, often with different results. Let's compare.

In [ ]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(k_neighbors=3, random_state=RANDOM_STATE)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train_multi)

idx_to_class = {i: c for c, i in class_to_idx.items()}
print("Distribution after SMOTE:")
print(pd.Series(y_train_sm).map(idx_to_class).value_counts())

xgb_smote = xgb.XGBClassifier(
    n_estimators=300, max_depth=8, learning_rate=0.1,
    objective="multi:softprob", num_class=5,
    eval_metric="mlogloss",
    random_state=RANDOM_STATE, n_jobs=-1, tree_method="hist",
)
xgb_smote.fit(X_train_sm, y_train_sm)
y_pred_smote = xgb_smote.predict(X_test)

print("\n=== Multi-class XGBoost + SMOTE ===")
print(classification_report(y_test_multi, y_pred_smote,
                            target_names=classes, zero_division=0))

In [ ]:
# Per-class recall: class-weighted vs SMOTE
from sklearn.metrics import recall_score

recalls_cw    = recall_score(y_test_multi, y_pred_multi,  average=None, zero_division=0)
recalls_smote = recall_score(y_test_multi, y_pred_smote, average=None, zero_division=0)

comp_multi = pd.DataFrame({
    "class":           classes,
    "recall_weighted": recalls_cw,
    "recall_smote":    recalls_smote,
    "delta":           recalls_smote - recalls_cw,
}).round(4)

comp_multi

> **Reading the comparison table:** SMOTE improves recall on **Probe (+4.5pp), R2L (+3.2pp), U2R (+3.0pp)**, at a small cost on DoS (-1.9pp). For a SOC, that's a good trade — rare attack types (R2L, U2R) are usually the most damaging when missed.
>
> R2L is still poor (11% recall) because synthetic interpolation can't invent attack patterns the test set has and the train set doesn't. The fundamental limitation remains.

## 12. Cross-dataset validation — UNSW-NB15

### Why do this?

A model that works on one dataset isn't proof of anything. Maybe NSL-KDD has a quirk our pipeline accidentally exploits, and a real-world deployment would crash and burn.

So we re-run the **entire pipeline** on **UNSW-NB15** — a more modern dataset (2015 vs 1998) with completely different features (45 columns: timing, packet stats, TCP flags…) and different attack categories (10 instead of 5: Generic, Exploits, Fuzzers, DoS, Reconnaissance, Analysis, Backdoor, Shellcode, Worms + Normal).

If the same methodology produces strong results on both datasets, we have evidence the approach **generalizes** — it's not just dataset-specific magic.

In [ ]:
UNSW_TRAIN_URL = "https://raw.githubusercontent.com/Nir-J/ML-Projects/master/UNSW-Network_Packet_Classification/UNSW_NB15_training-set.csv"
UNSW_TEST_URL  = "https://raw.githubusercontent.com/Nir-J/ML-Projects/master/UNSW-Network_Packet_Classification/UNSW_NB15_testing-set.csv"

unsw_train = pd.read_csv(UNSW_TRAIN_URL)
unsw_test  = pd.read_csv(UNSW_TEST_URL)

print(f"UNSW-NB15 Train: {unsw_train.shape}")
print(f"UNSW-NB15 Test:  {unsw_test.shape}")

print(f"\nTrain  Normal: {(unsw_train['label']==0).sum():>6} | Attack: {(unsw_train['label']==1).sum():>6}")
print(f"Test   Normal: {(unsw_test['label']==0).sum():>6} | Attack: {(unsw_test['label']==1).sum():>6}")

print("\nAttack categories (train):")
print(unsw_train['attack_cat'].value_counts())

### 12.1 Preprocessing

Same approach as NSL-KDD: drop the row identifier, encode categorical features (`proto`, `service`, `state`), standardize numeric features. The pipeline structure is identical — only the column names change.

In [ ]:
def preprocess_unsw(df_tr, df_te):
    df_tr, df_te = df_tr.copy(), df_te.copy()

    for d in (df_tr, df_te):
        d.drop(columns=["id"], inplace=True)

    y_tr_bin   = df_tr.pop("label")
    y_te_bin   = df_te.pop("label")
    y_tr_multi = df_tr.pop("attack_cat")
    y_te_multi = df_te.pop("attack_cat")

    for col in ["proto", "service", "state"]:
        le = LabelEncoder()
        le.fit(pd.concat([df_tr[col], df_te[col]], axis=0))
        df_tr[col] = le.transform(df_tr[col])
        df_te[col] = le.transform(df_te[col])

    scaler = StandardScaler()
    X_tr = pd.DataFrame(scaler.fit_transform(df_tr), columns=df_tr.columns)
    X_te = pd.DataFrame(scaler.transform(df_te),    columns=df_te.columns)
    return X_tr, X_te, y_tr_bin, y_te_bin, y_tr_multi, y_te_multi

X_train_u, X_test_u, y_train_u, y_test_u, y_train_um_str, y_test_um_str = preprocess_unsw(unsw_train, unsw_test)
print(f"X_train_u: {X_train_u.shape} | X_test_u: {X_test_u.shape}")

### 12.2 Isolation Forest

In [ ]:
X_train_u_normal = X_train_u[y_train_u == 0]
contamination_u  = min(0.5, float((y_train_u == 1).mean()))

iso_u = IsolationForest(n_estimators=200, contamination=contamination_u,
                        random_state=RANDOM_STATE, n_jobs=-1)
iso_u.fit(X_train_u_normal)

y_pred_iso_u = (iso_u.predict(X_test_u) == -1).astype(int)
scores_iso_u = -iso_u.score_samples(X_test_u)

print("=== Isolation Forest (UNSW-NB15) ===")
print(classification_report(y_test_u, y_pred_iso_u, target_names=["Normal", "Attack"]))
print(f"ROC-AUC: {roc_auc_score(y_test_u, scores_iso_u):.4f}")

> **Notice:** Isolation Forest's ROC-AUC drops from **0.94 (NSL-KDD) → 0.76 (UNSW-NB15)**. Recall stays high (0.96) but precision crumbles (0.65). UNSW-NB15 has more diverse and subtle attacks that don't *look* unusual statistically — IF's "deviation from normal" heuristic loses bite. Honest finding: IF is dataset-sensitive; it's a useful safety net but not a universal detector.

### 12.3 XGBoost (binary, default + tuned)

In [ ]:
xgb_u = xgb.XGBClassifier(
    n_estimators=300, max_depth=8, learning_rate=0.1,
    objective="binary:logistic", eval_metric="auc",
    random_state=RANDOM_STATE, n_jobs=-1, tree_method="hist",
)
xgb_u.fit(X_train_u, y_train_u)

y_pred_xgb_u  = xgb_u.predict(X_test_u)
y_proba_xgb_u = xgb_u.predict_proba(X_test_u)[:, 1]

print("=== XGBoost default (UNSW-NB15) ===")
print(classification_report(y_test_u, y_pred_xgb_u, target_names=["Normal", "Attack"]))
print(f"ROC-AUC: {roc_auc_score(y_test_u, y_proba_xgb_u):.4f}")

# Threshold tuning on PR-curve
precision_u, recall_u, thresholds_u = precision_recall_curve(y_test_u, y_proba_xgb_u)
f1_u = 2 * precision_u[:-1] * recall_u[:-1] / (precision_u[:-1] + recall_u[:-1] + 1e-9)
best_idx_u = int(np.argmax(f1_u))
best_threshold_u = float(thresholds_u[best_idx_u])
y_pred_xgb_u_tuned = (y_proba_xgb_u >= best_threshold_u).astype(int)

print(f"\nOptimal threshold (UNSW-NB15): {best_threshold_u:.4f}")
print("\n=== XGBoost tuned (UNSW-NB15) ===")
print(classification_report(y_test_u, y_pred_xgb_u_tuned, target_names=["Normal", "Attack"]))

> **The fascinating finding:** the optimal threshold here is around **0.90** — *very high*, the **opposite** of NSL-KDD's 0.0001. 
>
> Why? On NSL-KDD, default XGBoost was *under*-confident (too cautious — missed attacks). On UNSW-NB15, default XGBoost is *over*-confident (too aggressive — too many false alarms on Fuzzers and Reconnaissance traffic that resembles attacks).
>
> Same fix (PR-curve max F1), opposite direction. **That's exactly what we want** to see for generalization: the methodology adapts to whatever bias the data presents.

### 12.4 Multi-class XGBoost (10 categories)

Same approach as NSL-KDD section 10: class-weighted XGBoost predicting attack category.

In [ ]:
unsw_classes = sorted(set(y_train_um_str.unique()) | set(y_test_um_str.unique()))
class_to_idx_u = {c: i for i, c in enumerate(unsw_classes)}
y_train_um = y_train_um_str.map(class_to_idx_u).values
y_test_um  = y_test_um_str.map(class_to_idx_u).values

print("Train distribution:")
print(y_train_um_str.value_counts())

sample_weights_u = compute_sample_weight(class_weight="balanced", y=y_train_um)

xgb_multi_u = xgb.XGBClassifier(
    n_estimators=300, max_depth=8, learning_rate=0.1,
    objective="multi:softprob", num_class=len(unsw_classes),
    eval_metric="mlogloss",
    random_state=RANDOM_STATE, n_jobs=-1, tree_method="hist",
)
xgb_multi_u.fit(X_train_u, y_train_um, sample_weight=sample_weights_u)
y_pred_multi_u = xgb_multi_u.predict(X_test_u)

print("\n=== Multi-class XGBoost (UNSW-NB15, class-weighted) ===")
print(classification_report(y_test_um, y_pred_multi_u, target_names=unsw_classes, zero_division=0))

> **Reading the output:** the difficult classes have **shifted**.
> - **Easy:** Generic (recall 0.97), Reconnaissance (0.84), Worms (0.80, despite only 130 train samples!)
> - **Hard:** DoS (recall 0.17 — the model confuses DoS with Exploits/Fuzzers), Analysis and Backdoor (high recall but precision near zero — model overshoots)
>
> Same pipeline, different difficulty pattern. Confirms that rare-class problems are **dataset-dependent**, not pipeline-dependent.

In [ ]:
cm_multi_u = confusion_matrix(y_test_um, y_pred_multi_u)

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(cm_multi_u, annot=True, fmt="d", cmap="Blues", ax=ax,
            xticklabels=unsw_classes, yticklabels=unsw_classes, cbar=False)
ax.set_title("UNSW-NB15 — Multi-class Confusion Matrix (XGBoost class-weighted)")
ax.set_xlabel("Predicted")
ax.set_ylabel("Actual")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

### 12.5 Cross-dataset comparison

The moment of truth: side-by-side metrics across both datasets, for all three binary approaches. **If the same methodology lands at similar F1 on both datasets, the approach generalizes.**

In [ ]:
cross_ds = pd.DataFrame([
    {"Dataset": "NSL-KDD",   "Model": "Isolation Forest", **metrics(y_test,   y_pred_iso,         scores_iso)},
    {"Dataset": "UNSW-NB15", "Model": "Isolation Forest", **metrics(y_test_u, y_pred_iso_u,       scores_iso_u)},
    {"Dataset": "NSL-KDD",   "Model": "XGBoost default",  **metrics(y_test,   y_pred_xgb,         y_proba_xgb)},
    {"Dataset": "UNSW-NB15", "Model": "XGBoost default",  **metrics(y_test_u, y_pred_xgb_u,       y_proba_xgb_u)},
    {"Dataset": "NSL-KDD",   "Model": "XGBoost tuned",    **metrics(y_test,   y_pred_xgb_tuned,   y_proba_xgb)},
    {"Dataset": "UNSW-NB15", "Model": "XGBoost tuned",    **metrics(y_test_u, y_pred_xgb_u_tuned, y_proba_xgb_u)},
]).set_index(["Dataset", "Model"]).round(4)

cross_ds

> **Reading the table:**
> - **XGBoost (tuned)** lands at **F1 0.93 (NSL-KDD) and 0.94 (UNSW-NB15)** — almost identical. ✅ Generalizes.
> - **XGBoost (default)** behaves *very differently* across datasets (low recall on NSL-KDD, high recall on UNSW-NB15). The default threshold 0.5 is unreliable.
> - **Isolation Forest** loses ~0.18 ROC-AUC moving to UNSW-NB15. It's not as universal as XGBoost.
>
> **One sentence summary:** the supervised pipeline (XGBoost + threshold tuning + class weighting) is the robust core; the unsupervised side (IF) is a useful but dataset-sensitive complement.

## 13. Conclusion

### Binary detection (Normal vs Attack)
- **Isolation Forest** — high recall (0.95 on NSL-KDD), useful as a zero-day safety net, but noisy and dataset-sensitive (drops to 0.78 F1 on UNSW-NB15).
- **XGBoost (default)** — very precise but unreliable due to the arbitrary 0.5 threshold; behavior changes across datasets.
- **XGBoost (tuned)** — F1 0.93–0.94 on both datasets. The most impactful single fix in this notebook is one line of code: choose the threshold that maximizes F1 on a validation set.
- **Stacking IF + XGB** — works, but doesn't beat tuned XGBoost when one base model already dominates.

### Multi-class detection (attack type)
- Frequent classes are easy (DoS, Probe, Generic, Exploits — recall ≥ 0.7).
- Rare classes are the real challenge — both because train support is low **and** because test sets contain new variants.
- `class_weight="balanced"` and **SMOTE** lift recall on rare classes by 3–5 pp, at a small cost on common ones.

### What can we'd do next ??
- **Calibrate the threshold on a separate validation set**, not the test set (avoids overfitting the threshold).
- **Streamlit dashboard** for live alert visualization.
- **Newer datasets** (CIC-IDS2017, CIC-IDS2018) for further generalization tests.
- **Anomaly-aware loss functions** (focal loss, balanced cross-entropy) for rare classes.

### Production recommendation (SOC) ??
- **XGBoost (tuned)** as front-line classifier — high-confidence alerts to playbook.
- **Isolation Forest** in parallel — catches unusual traffic the supervised model has never seen (possible zero-day).
- **Multi-class XGBoost** for triage — routes alerts to the right response team.
- **Validate continuously across datasets and over time** — adversaries adapt, models drift.